In [ ]:
import gc
import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
import torch
import torch.nn as nn

%matplotlib inline
plt.rcParams["figure.dpi"] = 120
sns.set_theme(style="whitegrid")

# Φόρτωση όλων των αποθηκευμένων αποτελεσμάτων από Ερωτήματα 3–6
q34_df = pd.read_csv("results/model_comparison.csv")
mlp_bin_sum = pd.read_csv("results/mlp_final_summary.csv").iloc[0]
q5_df = pd.read_csv("results/q5_feature_pruning_results.csv")
q6_df = pd.read_csv("results/q6_multiclass_model_comparison.csv")

# Μέτρηση XAI Latency και για το Binary MLP (DeepExplainer, bg=100, ίδιο πρωτόκολλο με Q6)
class IntrusionDetectionMLP(nn.Module):
    def __init__(self, input_features=44):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_features, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        return self.network(x)

cols = pd.read_csv("processed/train_final_scaled.csv", nrows=0).columns.tolist()
features = [c for c in cols if c != "Label"]
dtypes = {c: "float32" for c in features}
dtypes["Label"] = "int16"

tr_sample = pd.read_csv("processed/train_final_scaled.csv", dtype=dtypes, nrows=50000)
te_sample = pd.read_csv("processed/test_final_scaled.csv", dtype=dtypes, nrows=5000)
rng = np.random.default_rng(42)
bg_idx = rng.choice(len(tr_sample), size=100, replace=False)
X_bg_t = torch.from_numpy(np.ascontiguousarray(tr_sample.iloc[bg_idx][features].to_numpy(dtype=np.float32)))
X_te_t = torch.from_numpy(np.ascontiguousarray(te_sample[features].to_numpy(dtype=np.float32)))
del tr_sample, te_sample
gc.collect()

mlp_bin = IntrusionDetectionMLP(len(features))
mlp_bin.load_state_dict(torch.load("models/mlp_binary_best.pt", map_location="cpu", weights_only=True))
mlp_bin.eval()

mlp_bin_exp = shap.DeepExplainer(mlp_bin, X_bg_t)
for _ in range(5):
    mlp_bin_exp.shap_values(X_te_t[:1], check_additivity=False)
mlp_bin_xai_single = []
for i in range(100):
    ts = time.perf_counter_ns()
    mlp_bin_exp.shap_values(X_te_t[i:i+1], check_additivity=False)
    mlp_bin_xai_single.append((time.perf_counter_ns() - ts) / 1_000_000)

XAI_BATCH = 64
mlp_bin_xai_bt = []
for _ in range(3):
    ts = time.perf_counter()
    mlp_bin_exp.shap_values(X_te_t[:XAI_BATCH], check_additivity=False)
    mlp_bin_xai_bt.append(time.perf_counter() - ts)

mlp_bin_xai_med_ms = float(np.median(mlp_bin_xai_single))
mlp_bin_xai_thr = float(XAI_BATCH / np.median(mlp_bin_xai_bt))
print(f"Binary MLP DeepExplainer (bg=100): Single Med={mlp_bin_xai_med_ms:.3f} ms | Batch XAI Thr={mlp_bin_xai_thr:.1f} flows/s")

# Σύνθεση ενοποιημένου πίνακα για όλα τα 6 μοντέλα (Binary & Multiclass)
rf_bin_q3 = q34_df[q34_df["Model"] == "Random Forest"].iloc[0]
km_bin_q3 = q34_df[q34_df["Model"] == "K-Means"].iloc[0]
mlp_bin_q3 = q34_df[q34_df["Model"] == "MLP"].iloc[0]
rf_q5_44 = q5_df[q5_df["Num_Features"] == 44].iloc[0]
rf_q5_10 = q5_df[q5_df["Num_Features"] == 10].iloc[0]
rf_mc_q6 = q6_df[q6_df["Model"].str.contains("Random Forest")].iloc[0]
mlp_mc_q6 = q6_df[q6_df["Model"].str.contains("MLP")].iloc[0]

rf_bin_size_kb = Path("models/rf_binary_44.joblib").stat().st_size / 1024.0
mlp_bin_comb_thr = 1000.0 / ((1000.0 / float(mlp_bin_q3["Throughput (flows/s)"])) + (1000.0 / mlp_bin_xai_thr))

models_rows = [
    {
        "Model": "Binary RF (44 feat.)",
        "Task": "Binary (ML)",
        "Accuracy": float(rf_bin_q3["Accuracy"]),
        "Primary_F1": float(rf_bin_q3["F1"]),
        "Attack_Recall": float(rf_bin_q3["Recall"]),
        "FPR": float(rf_bin_q3["FPR"]),
        "Model_Size_KB": rf_bin_size_kb,
        "Single_Inf_Med_ms": float(rf_bin_q3["Median Latency (ms)"]),
        "Single_Inf_P95_ms": float(rf_bin_q3["P95 Latency (ms)"]),
        "Seq_Max_Rate_fps": 1000.0 / float(rf_bin_q3["Median Latency (ms)"]),
        "Batch_Max_Rate_fps": float(rf_bin_q3["Throughput (flows/s)"]),
        "Batch_Amortized_us": 1_000_000.0 / float(rf_bin_q3["Throughput (flows/s)"]),
        "XAI_Method": "TreeExplainer",
        "XAI_Single_Med_ms": float(rf_q5_44["XAI_Median_ms"]),
        "Total_Single_Inf_XAI_ms": float(rf_bin_q3["Median Latency (ms)"]) + float(rf_q5_44["XAI_Median_ms"]),
        "Inline_XAI_Batch_fps": float(rf_q5_44["Combined_Batch_Throughput_fps"]),
    },
    {
        "Model": "Binary RF (Top-10)",
        "Task": "Binary Pruned (ML)",
        "Accuracy": float(rf_q5_10["Accuracy"]),
        "Primary_F1": float(rf_q5_10["F1"]),
        "Attack_Recall": float(rf_q5_10["Recall"]),
        "FPR": float(rf_q5_10["FPR"]),
        "Model_Size_KB": float(rf_q5_10["Model_Size_KB"]),
        "Single_Inf_Med_ms": float(rf_q5_10["Inf_Median_ms"]),
        "Single_Inf_P95_ms": float(rf_q5_10["Inf_P95_ms"]),
        "Seq_Max_Rate_fps": 1000.0 / float(rf_q5_10["Inf_Median_ms"]),
        "Batch_Max_Rate_fps": float(rf_q5_10["Inf_Batch_Throughput_fps"]),
        "Batch_Amortized_us": 1_000_000.0 / float(rf_q5_10["Inf_Batch_Throughput_fps"]),
        "XAI_Method": "TreeExplainer",
        "XAI_Single_Med_ms": float(rf_q5_10["XAI_Median_ms"]),
        "Total_Single_Inf_XAI_ms": float(rf_q5_10["Total_Single_Median_ms"]),
        "Inline_XAI_Batch_fps": float(rf_q5_10["Combined_Batch_Throughput_fps"]),
    },
    {
        "Model": "Binary K-Means",
        "Task": "Binary (Unsup.)",
        "Accuracy": float(km_bin_q3["Accuracy"]),
        "Primary_F1": float(km_bin_q3["F1"]),
        "Attack_Recall": float(km_bin_q3["Recall"]),
        "FPR": float(km_bin_q3["FPR"]),
        "Model_Size_KB": 1.5,
        "Single_Inf_Med_ms": float(km_bin_q3["Median Latency (ms)"]),
        "Single_Inf_P95_ms": float(km_bin_q3["P95 Latency (ms)"]),
        "Seq_Max_Rate_fps": 1000.0 / float(km_bin_q3["Median Latency (ms)"]),
        "Batch_Max_Rate_fps": float(km_bin_q3["Throughput (flows/s)"]),
        "Batch_Amortized_us": 1_000_000.0 / float(km_bin_q3["Throughput (flows/s)"]),
        "XAI_Method": "N/A",
        "XAI_Single_Med_ms": np.nan,
        "Total_Single_Inf_XAI_ms": np.nan,
        "Inline_XAI_Batch_fps": np.nan,
    },
    {
        "Model": "Binary MLP",
        "Task": "Binary (DL)",
        "Accuracy": float(mlp_bin_q3["Accuracy"]),
        "Primary_F1": float(mlp_bin_q3["F1"]),
        "Attack_Recall": float(mlp_bin_q3["Recall"]),
        "FPR": float(mlp_bin_q3["FPR"]),
        "Model_Size_KB": float(mlp_bin_sum["Model Size (KB)"]),
        "Single_Inf_Med_ms": float(mlp_bin_q3["Median Latency (ms)"]),
        "Single_Inf_P95_ms": float(mlp_bin_q3["P95 Latency (ms)"]),
        "Seq_Max_Rate_fps": 1000.0 / float(mlp_bin_q3["Median Latency (ms)"]),
        "Batch_Max_Rate_fps": float(mlp_bin_q3["Throughput (flows/s)"]),
        "Batch_Amortized_us": 1_000_000.0 / float(mlp_bin_q3["Throughput (flows/s)"]),
        "XAI_Method": "DeepExplainer (bg=100)",
        "XAI_Single_Med_ms": mlp_bin_xai_med_ms,
        "Total_Single_Inf_XAI_ms": float(mlp_bin_q3["Median Latency (ms)"]) + mlp_bin_xai_med_ms,
        "Inline_XAI_Batch_fps": mlp_bin_comb_thr,
    },
    {
        "Model": "Multiclass RF (11 cl.)",
        "Task": "Multiclass (ML)",
        "Accuracy": float(rf_mc_q6["Accuracy"]),
        "Primary_F1": float(rf_mc_q6["Macro_F1"]),
        "Attack_Recall": float(rf_mc_q6["Binary_Attack_Recall"]),
        "FPR": float(rf_mc_q6["Binary_FPR"]),
        "Model_Size_KB": float(rf_mc_q6["Model_Size_KB"]),
        "Single_Inf_Med_ms": float(rf_mc_q6["Inf_Median_ms"]),
        "Single_Inf_P95_ms": float(rf_mc_q6["Inf_P95_ms"]),
        "Seq_Max_Rate_fps": 1000.0 / float(rf_mc_q6["Inf_Median_ms"]),
        "Batch_Max_Rate_fps": float(rf_mc_q6["Inf_Batch_Throughput_fps"]),
        "Batch_Amortized_us": 1_000_000.0 / float(rf_mc_q6["Inf_Batch_Throughput_fps"]),
        "XAI_Method": "TreeExplainer",
        "XAI_Single_Med_ms": float(rf_mc_q6["XAI_Median_ms"]),
        "Total_Single_Inf_XAI_ms": float(rf_mc_q6["Total_Inf_Plus_XAI_ms"]),
        "Inline_XAI_Batch_fps": float(rf_mc_q6["Combined_Batch_Throughput_fps"]),
    },
    {
        "Model": "Multiclass MLP (11 cl.)",
        "Task": "Multiclass (DL)",
        "Accuracy": float(mlp_mc_q6["Accuracy"]),
        "Primary_F1": float(mlp_mc_q6["Macro_F1"]),
        "Attack_Recall": float(mlp_mc_q6["Binary_Attack_Recall"]),
        "FPR": float(mlp_mc_q6["Binary_FPR"]),
        "Model_Size_KB": float(mlp_mc_q6["Model_Size_KB"]),
        "Single_Inf_Med_ms": float(mlp_mc_q6["Inf_Median_ms"]),
        "Single_Inf_P95_ms": float(mlp_mc_q6["Inf_P95_ms"]),
        "Seq_Max_Rate_fps": 1000.0 / float(mlp_mc_q6["Inf_Median_ms"]),
        "Batch_Max_Rate_fps": float(mlp_mc_q6["Inf_Batch_Throughput_fps"]),
        "Batch_Amortized_us": 1_000_000.0 / float(mlp_mc_q6["Inf_Batch_Throughput_fps"]),
        "XAI_Method": "DeepExplainer (bg=100)",
        "XAI_Single_Med_ms": float(mlp_mc_q6["XAI_Median_ms"]),
        "Total_Single_Inf_XAI_ms": float(mlp_mc_q6["Total_Inf_Plus_XAI_ms"]),
        "Inline_XAI_Batch_fps": float(mlp_mc_q6["Combined_Batch_Throughput_fps"]),
    },
]

unified_df = pd.DataFrame(models_rows)
unified_df.to_csv("results/q7_unified_models_table.csv", index=False)

print("\n=== UNIFIED 5G IDS PERFORMANCE & THROUGHPUT TABLE ===")
disp_cols = [
    "Model", "Primary_F1", "FPR", "Model_Size_KB",
    "Single_Inf_Med_ms", "Seq_Max_Rate_fps", "Batch_Amortized_us", "Batch_Max_Rate_fps",
    "XAI_Single_Med_ms", "Inline_XAI_Batch_fps"
]
print(unified_df[disp_cols].round(5).to_string(index=False))


In [ ]:
# Υπολογισμός ψευδών συναγερμών (False Alerts) και Operational Precision σε 5G σενάρια
BENIGN_SHARE = 0.99  # 99% κανονική κίνηση σε ρεαλιστικό δίκτυο παραγωγής

def operational_precision(recall, fpr, p_attack):
    tp_rate = p_attack * recall
    fp_rate = (1.0 - p_attack) * fpr
    return tp_rate / (tp_rate + fp_rate) if (tp_rate + fp_rate) > 0 else 0.0

fpr_rows = []
for _, r in unified_df.iterrows():
    fpr = r["FPR"]
    rec = r["Attack_Recall"]
    fa_10k_min = 10_000 * BENIGN_SHARE * 60 * fpr
    fa_10k_hr = 10_000 * BENIGN_SHARE * 3600 * fpr
    fa_100k_hr = 100_000 * BENIGN_SHARE * 3600 * fpr
    fa_1m_hr = 1_000_000 * BENIGN_SHARE * 3600 * fpr

    # Πραγματική ακρίβεια συναγερμών (ποιο % των alerts είναι αληθινές επιθέσεις)
    op_prec_01 = operational_precision(rec, fpr, 0.001) * 100.0   # 0.1% επιθέσεις
    op_prec_1 = operational_precision(rec, fpr, 0.01) * 100.0     # 1.0% επιθέσεις
    op_prec_ds = operational_precision(rec, fpr, 0.1209) * 100.0  # 12.09% (dataset)

    # Δυνατότητα εξυπηρέτησης on-alert XAI σε 10k flows/s με 1% επιθέσεις (~100 true attacks/s + false alerts/s)
    alerts_per_sec_10k_1pct = 10_000 * (0.01 * rec + 0.99 * fpr)
    xai_cap = r["Inline_XAI_Batch_fps"]
    can_explain_on_alert = (xai_cap >= alerts_per_sec_10k_1pct) if pd.notna(xai_cap) else False

    fpr_rows.append({
        "Model": r["Model"],
        "FPR_Pct": fpr * 100.0,
        "False_Alerts_per_Min_10k": fa_10k_min,
        "False_Alerts_per_Hr_10k": fa_10k_hr,
        "False_Alerts_per_Hr_100k": fa_100k_hr,
        "False_Alerts_per_Hr_1M": fa_1m_hr,
        "Op_Precision_0.1pct_Atk": op_prec_01,
        "Op_Precision_1pct_Atk": op_prec_1,
        "Op_Precision_12.1pct_Atk": op_prec_ds,
        "Total_Alerts_per_s_10k_1pct": alerts_per_sec_10k_1pct,
        "XAI_Capacity_alerts_per_s": xai_cap,
        "On_Alert_XAI_Feasible_10k": can_explain_on_alert,
    })

fpr_df = pd.DataFrame(fpr_rows)
fpr_df.to_csv("results/q7_fpr_operator_load.csv", index=False)

print("=== FALSE-POSITIVE OPERATOR BURDEN & OPERATIONAL PRECISION ===")
print(fpr_df[[
    "Model", "FPR_Pct", "False_Alerts_per_Min_10k", "False_Alerts_per_Hr_10k",
    "False_Alerts_per_Hr_100k", "Op_Precision_0.1pct_Atk", "Op_Precision_1pct_Atk"
]].round(3).to_string(index=False))

print("\n=== ON-ALERT XAI FEASIBILITY (10,000 flows/s link, 1% attack rate) ===")
print(fpr_df[[
    "Model", "Total_Alerts_per_s_10k_1pct", "XAI_Capacity_alerts_per_s", "On_Alert_XAI_Feasible_10k"
]].round(2).to_string(index=False))

# Αποθήκευση JSON σύνοψης Ερωτήματος 7
summary_q7 = {
    "traffic_scenarios_fps": {"edge_mec": 10_000, "mid_upf": 100_000, "core_link": 1_000_000},
    "benign_share_assumption": BENIGN_SHARE,
    "unified_models": unified_df.where(pd.notna(unified_df), None).to_dict(orient="records"),
    "fpr_operator_analysis": fpr_df.where(pd.notna(fpr_df), None).to_dict(orient="records"),
}
with open("results/q7_5g_analysis_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary_q7, f, indent=2, ensure_ascii=False)
print("\nSaved: results/q7_unified_models_table.csv, results/q7_fpr_operator_load.csv & results/q7_5g_analysis_summary.json")


In [ ]:
# --- Plot 1: Single-Flow Latency & Batch Throughput vs 5G Line Rates ---
fig, axes = plt.subplots(1, 2, figsize=(15.5, 6.0))
models = unified_df["Model"].tolist()
y_pos = np.arange(len(models))
h = 0.36

# (a) Single-flow latency
inf_ms = unified_df["Single_Inf_Med_ms"].to_numpy()
tot_ms = unified_df["Total_Single_Inf_XAI_ms"].to_numpy()

axes[0].barh(y_pos - h/2, inf_ms, height=h, label="Inference μόνο (Median ms)", color="#1f77b4")
xai_mask = ~np.isnan(tot_ms)
axes[0].barh(y_pos[xai_mask] + h/2, tot_ms[xai_mask], height=h, label="Inference + Inline SHAP (ms)", color="#d62728", alpha=0.88)
axes[0].set_xscale("log")
axes[0].set_yticks(y_pos)
axes[0].set_yticklabels(models, fontsize=9.5)
axes[0].invert_yaxis()
axes[0].set_xlabel("Single-Flow Latency (ms — λογαριθμική κλίμακα)")
axes[0].set_title("(α) Χρόνος Inference ανά Ροή: Χωρίς vs Με Inline XAI", fontweight="bold")
for i in range(len(models)):
    axes[0].text(inf_ms[i] * 1.18, y_pos[i] - h/2, f"{inf_ms[i]:.3f} ms", va="center", fontsize=8, color="#1f77b4", fontweight="bold")
    if not np.isnan(tot_ms[i]):
        axes[0].text(tot_ms[i] * 1.18, y_pos[i] + h/2, f"{tot_ms[i]:.2f} ms", va="center", fontsize=8, color="#b22222", fontweight="bold")
axes[0].set_xlim(0.02, 260)
axes[0].legend(loc="upper right", fontsize=9, frameon=True)
axes[0].grid(axis="x", which="both", linestyle="--", alpha=0.45)

# (b) Batch Throughput vs 5G targets
thr_inf = unified_df["Batch_Max_Rate_fps"].to_numpy()
thr_xai = unified_df["Inline_XAI_Batch_fps"].to_numpy()

axes[1].barh(y_pos - h/2, thr_inf, height=h, label="Inference Batch Throughput (flows/s)", color="#2ca02c")
xai_tmask = ~np.isnan(thr_xai)
axes[1].barh(y_pos[xai_tmask] + h/2, thr_xai[xai_tmask], height=h, label="Inference + Inline XAI (flows/s)", color="#ff7f0e")
axes[1].set_xscale("log")
axes[1].set_yticks(y_pos)
axes[1].set_yticklabels([""] * len(models))
axes[1].invert_yaxis()
axes[1].set_xlabel("Μέγιστος Ρυθμός Επεξεργασίας (flows/s — log scale)")
axes[1].set_title("(β) Μέγιστο Throughput έναντι Τυπικών Φορτίων 5G", fontweight="bold")

for rate, lbl, col in [(10_000, "5G Edge (10k)", "#555555"), (100_000, "5G UPF (100k)", "#6a3d9a"), (1_000_000, "5G Core (1M)", "#b15928")]:
    axes[1].axvline(rate, color=col, linestyle="--", linewidth=1.3, alpha=0.8)
    axes[1].text(rate * 1.08, -0.42, lbl, color=col, fontsize=8, fontweight="bold", va="bottom")

for i in range(len(models)):
    axes[1].text(thr_inf[i] * 1.18, y_pos[i] - h/2, f"{thr_inf[i]:,.0f}", va="center", fontsize=8, color="#1f7a1f", fontweight="bold")
    if not np.isnan(thr_xai[i]):
        axes[1].text(thr_xai[i] * 1.18, y_pos[i] + h/2, f"{thr_xai[i]:,.0f}", va="center", fontsize=8, color="#cc5500", fontweight="bold")

axes[1].set_xlim(10, 3e7)
axes[1].legend(loc="lower right", fontsize=9, frameon=True)
axes[1].grid(axis="x", which="both", linestyle="--", alpha=0.45)

plt.tight_layout()
plt.savefig("results/q7_latency_throughput_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

# --- Plot 2: FPR Operator Alert Burden & Operational Precision ---
fig, axes = plt.subplots(1, 2, figsize=(15.5, 5.8))

# (a) False alerts per hour at 10,000 flows/s (log scale)
sorted_fpr = fpr_df.sort_values("False_Alerts_per_Hr_10k", ascending=True)
colors_bar = ["#2ca02c" if v < 1500 else "#ff7f0e" if v < 50000 else "#d62728" for v in sorted_fpr["False_Alerts_per_Hr_10k"]]
axes[0].barh(sorted_fpr["Model"], sorted_fpr["False_Alerts_per_Hr_10k"], color=colors_bar, alpha=0.88)
axes[0].set_xscale("log")
axes[0].set_xlabel("Ψευδείς Συναγερμοί / Ώρα (σε σύνδεση 10.000 flows/s, 99% Benign)")
axes[0].set_title("(α) Επιβάρυνση Network Operator από False Positives", fontweight="bold")
for i, (_, r) in enumerate(sorted_fpr.iterrows()):
    v = r["False_Alerts_per_Hr_10k"]
    axes[0].text(v * 1.2, i, f"{v:,.0f} / ώρα (FPR {r['FPR_Pct']:.4f}%)", va="center", fontsize=8.5, fontweight="bold")
axes[0].set_xlim(50, 2e8)
axes[0].grid(axis="x", which="both", linestyle="--", alpha=0.45)

# (b) Operational Precision vs Attack Prevalence
prevs = np.logspace(-4, -0.8, 120)  # 0.01% to ~15.8%
palette = ["#1f77b4", "#17becf", "#d62728", "#ff7f0e", "#2ca02c", "#9467bd"]
for idx, (_, r) in enumerate(unified_df.iterrows()):
    op_p = [operational_precision(r["Attack_Recall"], r["FPR"], p) * 100.0 for p in prevs]
    axes[1].plot(prevs * 100.0, op_p, linewidth=2.1, label=r["Model"], color=palette[idx])
axes[1].set_xscale("log")
axes[1].axvline(0.1, color="gray", linestyle=":", linewidth=1.2)
axes[1].text(0.11, 12, "0.1% επιθέσεις\n(τυπικό 5G)", fontsize=8.5, color="dimgray")
axes[1].set_xlabel("Ποσοστό Πραγματικών Επιθέσεων στη Δικτυακή Κίνηση (% — log scale)")
axes[1].set_ylabel("Operational Precision (% αληθινών alerts)")
axes[1].set_title("(β) Πραγματική Ακρίβεια Συναγερμών (Alert Fatigue Analysis)", fontweight="bold")
axes[1].set_ylim(-3, 105)
axes[1].legend(loc="lower right", fontsize=8.5, frameon=True)
axes[1].grid(True, which="both", linestyle="--", alpha=0.45)

plt.tight_layout()
plt.savefig("results/q7_fpr_alert_fatigue.png", dpi=150, bbox_inches="tight")
plt.show()

# --- Plot 3: 5G Multi-Objective Trade-off Map (Throughput vs FPR vs Model Size) ---
fig, ax = plt.subplots(figsize=(10.5, 6.2))
sup_df = unified_df[unified_df["Model"] != "Binary K-Means"].copy()
sizes_plot = np.sqrt(sup_df["Model_Size_KB"].to_numpy()) * 16 + 70
colors_sc = ["#1f77b4", "#17becf", "#ff7f0e", "#2ca02c", "#9467bd"]

for i, (_, r) in enumerate(sup_df.iterrows()):
    ax.scatter(
        r["Batch_Max_Rate_fps"], r["FPR"] * 100.0,
        s=sizes_plot[i], color=colors_sc[i], alpha=0.85, edgecolor="black", linewidth=1.1,
        label=f"{r['Model']} ({r['Model_Size_KB']:.0f} KB)"
    )
    if "Binary MLP" in r["Model"]:
        dx, dy, ha = 0, -26, "center"
    elif "Multiclass MLP" in r["Model"]:
        dx, dy, ha = -10, 12, "right"
    elif "44" in r["Model"]:
        dx, dy, ha = 14, 10, "left"
    elif "Top-10" in r["Model"]:
        dx, dy, ha = 14, -14, "left"
    else:
        dx, dy, ha = 14, 10, "left"
    ax.annotate(
        f"{r['Model']}\n({r['Model_Size_KB']:.0f} KB, XAI: {r['XAI_Single_Med_ms']:.1f} ms)",
        (r["Batch_Max_Rate_fps"], r["FPR"] * 100.0),
        xytext=(dx, dy), textcoords="offset points", ha=ha, fontsize=8.5, fontweight="bold"
    )

ax.set_xscale("log")
ax.set_yscale("log")
ax.invert_yaxis()  # Lower FPR = better (top-right = ideal corner!)
ax.set_xlabel("Μέγιστος Ρυθμός Επεξεργασίας — Batch Throughput (flows/s, log scale) → Καλύτερο")
ax.set_ylabel("False-Positive Rate (%, log scale — αντεστραμμένο) ↑ Καλύτερο")
ax.set_title("Χάρτης Trade-off 5G IDS: Throughput vs FPR vs Μέγεθος Μοντέλου (Μέγεθος Κύκλου)", fontweight="bold")
ax.text(0.97, 0.95, "Ιδανική Περιοχή 5G\n(Υψηλό Throughput & Ελάχιστο FPR)", transform=ax.transAxes,
        ha="right", va="top", fontsize=9, fontweight="bold", color="#1f7a1f",
        bbox=dict(boxstyle="round,pad=0.35", facecolor="#e8f5e9", edgecolor="#2ca02c"))
ax.set_xlim(4e4, 7e6)
ax.set_ylim(0.25, 0.00025)
ax.grid(True, which="both", linestyle="--", alpha=0.45)
plt.tight_layout()
plt.savefig("results/q7_tradeoff_pareto.png", dpi=150, bbox_inches="tight")
plt.show()
